In [26]:
import numpy as np
from scipy import signal as sig
import matplotlib.pyplot as plt
import Funciones as Funciones
from NNMF_GIF import NNMF_GIF 
from pathlib import Path
from ipyfilechooser import FileChooser
import ipywidgets as widgets
from IPython.display import  clear_output
%config InlineBackend.figure_format = 'svg'
import librosa as lib

In [27]:
FC = FileChooser('Señales VEC')
FC.layout = widgets.Layout(width='1000px')
display(FC)

FileChooser(path='G:\Mi unidad\Facultad\Beca CIN\NNMF-GIF\Señales VEC', filename='', title='', show_hidden=Fal…

In [ ]:
archivo = Path(FC.selected)
nombre_archivo = archivo.name
datos = np.loadtxt(archivo, delimiter='\t', skiprows=1)
#datos = datos[:,:-1]
linewidth = 2
alfa_grid = 0.3
fs_r = 8000
to = 0
tf = 50
tiempos       = np.arange(to,tf,1000/fs_r)
color_notch = 'red'
color_lpc  = 'orange'
color_flujo = 'k'
color1 = 'red'
color2 = 'blue'

señal_OVV_  = datos[:, 2] - np.mean(datos[:, 2])

señal_OVV_ = señal_OVV_/max(abs(señal_OVV_))
#---------------- Filtro pasa bajos en 1,1 kHz -------------------------------------------------------------------------
orden = 10 
ripple = 0.1
fc     = 1100
fs_r = 8000
b,a   = sig.cheby1(orden,ripple,fc,'lowpass',fs=fs_r)
señal_OVV_f = sig.filtfilt(b,a,señal_OVV_)
fft_flujo_OVV_original, frec_flujo_OVV_original= Funciones.FFT(señal_OVV_f, fs_r)
#---------------- Integro para sacar el efecto derivador que tiene la señal  ------------------------------------------
#alfa = 0.96
# Filtro según el paper de BROOKE
'''
alfa= np.exp(-2*np.pi*50/fs_r)
num = [1,-1]
den = [1,-2*alfa,alfa**2]
flujo_OVV_original =  sig.lfilter(num,den,señal_OVV_f)
maximo = max(abs(flujo_OVV_original))
flujo_OVV_original = [val/maximo for val in flujo_OVV_original]

fft_flujo_OVV_original, frec_flujo_OVV_original= Funciones.FFT(flujo_OVV_original, fs_r)
'''
#---------------- Aplico el Notch  -----------------------------------------------------------------------------------

flujo_ovv, FF,error,b_notch,a_notch = Funciones.INF(señal_OVV_f)
maximo = max(abs(flujo_ovv))
flujo_ovv = [val/maximo for val in flujo_ovv]
#------------------------------------------------------------------------------------------------------------------------------
Q_widget = widgets.FloatText(
    value=10,
    min=0.001,
    max=1000.00,
    step=0.1,
    description='Q =',
    disabled=False,
    style={'description_width':'initial'},
    layout=widgets.Layout(width='100px')
)
slider = widgets.IntSlider(
    value=FF,
    min=200,
    max=1000,
    step=1,
    description='Frecuencia [Hz]:',
    style={'description_width':'initial'},
)
reset_button = widgets.Button(
    description="Defecto",
    style={'description_width':'initial'},
    layout=widgets.Layout(width='100px')
)
guardar_button = widgets.Button(
    description="Guardar"
)
def reset_slider(b):
    slider.value = FF

reset_button.on_click(reset_slider)
def guardar(b):
    with open(archivo, 'r') as f:
        cabecera = f.readline().strip()
    if datos.shape[1] > 3:
        datos[:, 3] = flujo_ovv
        frec_ = np.full(len(flujo_ovv), np.nan)
        frec_[0] = slider.value
        datos[:, 4] = frec_
        Q_ = np.full(len(flujo_ovv), np.nan)
        Q_[0] = Q_widget.value
        datos[:, 5] = Q_
        np.savetxt(
            archivo,
            datos,
            fmt="%.6f",
            delimiter='\t',
            header=cabecera,
            comments=''
        )
    else:
        frec_ = np.full(len(flujo_ovv), np.nan)
        frec_[0] = slider.value
        Q_ = np.full(len(flujo_ovv), np.nan)
        Q_[0] = Q_widget.value
        datos_nuevos = np.column_stack((datos, flujo_ovv,frec_,Q_))
        cabecera_nueva = cabecera + '\tFlujo OVV' + '\tFrecuencia_Notch' + '\tQ Notch'
        np.savetxt(
            archivo,
            datos_nuevos,
            fmt="%.6f",
            delimiter='\t',
            header=cabecera_nueva,
            comments=''
        )

reset_button.on_click(reset_slider)
guardar_button.on_click(guardar)
label = widgets.Label(value=f"Ancho de banda= {round(FF/Q_widget.value,0)} [Hz]")
def actualizar_label(change):
    label.value = f"Ancho de banda= {round(FF/Q_widget.value,0)} [Hz]"
slider.observe(actualizar_label, names='value')
Q_widget.observe(actualizar_label, names='value')
 
def graficar(frec_slider,Q_):
    clear_output(wait=True)
    fig, ax = plt.subplots(1, 2, figsize=(13, 5), gridspec_kw={'width_ratios': [2, 1]})
    fig.subplots_adjust(hspace=0.25)
    ax1 = ax[0]
    ax2 = ax[1]
    ax1.clear()
    ax2.clear()
    b, a = sig.iirnotch(w0=frec_slider,fs=fs_r, Q=Q_)
    w_notch, h_notch = sig.freqz(a,b,fs=fs_r,worN=4096)
    global flujo_ovv
    flujo_ovv = sig.lfilter(b, a, señal_OVV_f)
    maximo = max(abs(flujo_ovv))
    flujo_ovv = [val/maximo for val in flujo_ovv]
    ax1.plot(tiempos, señal_OVV_f, color='k', label=r'$u_{g}(t)$',linewidth=linewidth)
    ax1.plot(tiempos, flujo_ovv, color=color_notch, label=r'$u_{g}(t)$ - Notch',linewidth=linewidth)
    ax1.legend(loc='upper right')
    ax1.set_xlabel('Tiempo [ms]')
    ax1.set_title(nombre_archivo,pad=20)
    ax1.set_xlim([10,30])
    ax2.plot(frec_flujo_OVV_original, 10*np.log10(abs(fft_flujo_OVV_original)), color=color_flujo, label=r'$u_{g}(t)$',linewidth=linewidth)
    ax2.plot(w_notch, 10*np.log10(abs(h_notch)), color=color_notch,label='Notch',linewidth=linewidth)
    ax2.legend(loc='center right')
    ax2.set_ylim([-10,20]);
    plt.show()
out = widgets.interactive_output(graficar, {'frec_slider': slider,'Q_':Q_widget})
ui = widgets.HBox([slider,reset_button, Q_widget,label, guardar_button]);
display(widgets.VBox([ui,out]));
